# 1194. Tournament Winners

## Problem
We need to determine the **winner in each group** of players.  
- The winner is the player who scored the **maximum total points** within their group.  
- In case of a tie, the player with the **lowest player_id** wins.  
- Players belong to groups, and all matches are played within the same group.  

---

## Schema

### Table: Players
| Column Name | Type | Description                  |
|-------------|------|------------------------------|
| player_id   | INT  | Primary key, unique player ID |
| group_id    | INT  | Group to which the player belongs |

---

### Table: Matches
| Column Name   | Type | Description                                      |
|---------------|------|--------------------------------------------------|
| match_id      | INT  | Primary key, unique match ID                     |
| first_player  | INT  | Player ID of the first player                    |
| second_player | INT  | Player ID of the second player                   |
| first_score   | INT  | Points scored by the first player                |
| second_score  | INT  | Points scored by the second player                |

---

## Sample Data

### Players
| player_id | group_id |
|-----------|----------|
| 15        | 1        |
| 25        | 1        |
| 30        | 1        |
| 45        | 1        |
| 10        | 2        |
| 35        | 2        |
| 50        | 2        |
| 20        | 3        |
| 40        | 3        |

### Matches
| match_id | first_player | second_player | first_score | second_score |
|----------|--------------|---------------|-------------|--------------|
| 1        | 15           | 45            | 3           | 0            |
| 2        | 30           | 25            | 1           | 2            |
| 3        | 30           | 15            | 2           | 0            |
| 4        | 40           | 20            | 5           | 2            |
| 5        | 35           | 50            | 1           | 1            |

---

## Expected Output
| group_id | player_id |
|----------|-----------|
| 1        | 15        |
| 2        | 35        |
| 3        | 40        |

---

## PySpark Code: Create DataFrames and Temp Views

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

# Schema for Players
players_schema = StructType([
    StructField("player_id", IntegerType(), False),
    StructField("group_id", IntegerType(), False)
])

# Schema for Matches
matches_schema = StructType([
    StructField("match_id", IntegerType(), False),
    StructField("first_player", IntegerType(), False),
    StructField("second_player", IntegerType(), False),
    StructField("first_score", IntegerType(), False),
    StructField("second_score", IntegerType(), False)
])

# Data for Players
players_data = [
    (15, 1),
    (25, 1),
    (30, 1),
    (45, 1),
    (10, 2),
    (35, 2),
    (50, 2),
    (20, 3),
    (40, 3)
]

# Data for Matches
matches_data = [
    (1, 15, 45, 3, 0),
    (2, 30, 25, 1, 2),
    (3, 30, 15, 2, 0),
    (4, 40, 20, 5, 2),
    (5, 35, 50, 1, 1)
]

# Create DataFrames
players_df = spark.createDataFrame(players_data, players_schema)
matches_df = spark.createDataFrame(matches_data, matches_schema)

# Register Temp Views
players_df.createOrReplaceTempView("Players")
matches_df.createOrReplaceTempView("Matches")

# Quick check
players_df.show()
matches_df.show()


In [0]:
%sql

WITH cte(SELECT match_id, first_player, first_score, sum(first_score) OVER (
			PARTITION BY first_player ORDER BY first_player ASC
			) AS total_fs FROM Matches),
	cte2(SELECT match_id, second_player, second_score, sum(second_score) OVER (
			PARTITION BY second_player ORDER BY second_player ASC
			) AS total_ss FROM Matches),
	cte3 AS (
		SELECT coalesce(first_player, second_player) AS player_id,
			(coalesce(first_score, 0) + coalesce(second_score, 0)) AS total_score_pp
		FROM cte
		INNER JOIN cte2
			ON cte.first_player = cte2.second_player
		
		UNION
		
		SELECT coalesce(first_player, second_player) AS player_id,
			(coalesce(first_score, 0) + coalesce(second_score, 0)) AS total_score_pp
		FROM cte
		LEFT JOIN cte2
			ON cte.first_player = cte2.second_player
		
		UNION
		
		SELECT coalesce(first_player, second_player) AS player_id,
			(coalesce(first_score, 0) + coalesce(second_score, 0)) AS total_score_pp
		FROM cte
		RIGHT JOIN cte2
			ON cte.first_player = cte2.second_player
		),
	cte4(SELECT c.player_id, sum(c.total_score_pp) AS total_score, p.group_id FROM cte3 c INNER JOIN Players p
		ON c.player_id = p.player_id GROUP BY c.player_id, p.group_id ORDER BY p.group_id DESC),
	cte5(SELECT total_score, row_number() OVER (
			PARTITION BY group_id ORDER BY total_score DESC,
				player_id ASC
			) AS rn, player_id, group_id FROM cte4)

SELECT group_id , player_id
FROM cte5
WHERE rn = 1


# Reflection on Tournament Winners Problem

## My Initial Thinking
1. I started by dividing the **Matches** table into two parts: one for `first_player` and one for `second_player`.  
2. I then joined these parts together so that scores could be collected into a single column instead of being split across two.  
3. After performing unions to combine the datasets, I aggregated scores using `SUM` and `GROUP BY`.  
4. Finally, I applied `ROW_NUMBER()` to filter out the player with the maximum score in each group.

---

## Mistakes in My First Approach
- The query became **too long and complex** because I relied on multiple joins and unions.  
- I overlooked a simpler way: using a **CASE statement** early in the query to handle whether a player was `first_player` or `second_player`.  
- The key detail is that the join condition should be based on `player_id = first_player OR player_id = second_player`.  
- Once that condition is set, the score can be calculated directly depending on which role the player had in the match.  
- This allows aggregation (`SUM` and `GROUP BY`) to happen in the same query, avoiding the need for multiple CTEs.  
- After that, a single additional CTE with `ROW_NUMBER()` is enough to determine the winner per group.

---

## Comparison of Approaches
- **First Query (long version):**  
  - Broke the problem into multiple CTEs (`cte`, `cte2`, `cte3`, etc.).  
  - Required unions and coalesce logic to merge scores.  
  - Produced correct results but was verbose and harder to follow.

- **Second Query (simplified version):**  
  - Used a `CASE` statement to directly assign scores based on whether the player was `first_player` or `second_player`.  
  - Joined `Players` and `Matches` once with an `OR` condition.  
  - Aggregated scores in the same step.  
  - Applied `ROW_NUMBER()` to pick the top scorer per group.  
  - Much shorter, cleaner, and aligned with the problem’s expectation.

---

## Key Learning
- Always look for opportunities to simplify queries by using **CASE statements** and direct joins instead of splitting logic into multiple parts.  
- Pay attention to the problem statement: the join condition here required an **OR** between `first_player` and `second_player`.  
- A concise query is easier to debug, maintain, and explain, while still producing the correct result.


In [0]:
%sql

WITH cte AS (
		SELECT sum(CASE 
					WHEN p.player_id = m.first_player
						THEN first_score
					WHEN p.player_id = m.second_player
						THEN second_score
					END) AS score,
			p.player_id,
			p.group_id
		FROM Players p
		INNER JOIN Matches m
			ON p.player_id = m.first_player
				OR p.player_id = m.second_player
		GROUP BY p.player_id,
			p.group_id
		),
	cte2(SELECT row_number() OVER (
			PARTITION BY group_id ORDER BY score DESC
			) rn, group_id, player_id FROM cte)

SELECT group_id,
	player_id
FROM cte2
WHERE rn = 1
order by group_id asc
